# 01. Setup & Data Generation
Star Schema の Fact / Dimension を生成して File Pruning を実測できる状態にする

<div style="background: linear-gradient(135deg, #FF3621 0%, #C0210F 100%); color: white; padding: 24px 32px; border-radius: 8px; margin-bottom: 24px;">
  <div style="font-size: 26px; font-weight: 700;">01. Setup & Data Generation</div>
  <div style="font-size: 15px; opacity: 0.9; margin-top: 6px;">Star Schema の Fact / Dimension を生成して File Pruning を実測できる状態にする</div>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin-bottom: 12px;">
  <div style="font-weight: 700; color: #0d47a1; margin-bottom: 6px;">📌 このノートブックの目的</div>
  <div style="color: #1a237e; line-height: 1.7;">
    お客様の質問「Star Schema で Dim を WHERE で絞ると、全ファイルスキャンされるのか？」を <b>実データで検証</b>するための土台を作ります。<br>
    <b>02_pruning_verification</b> ノートブックがこのノートブックで作成したテーブルを利用します。
  </div>
</div>

<div style="border-left: 4px solid #F57C00; background: #FFF3E0; padding: 16px 20px; border-radius: 4px;">
  <div style="font-weight: 700; color: #E65100; margin-bottom: 6px;">⚠️ 実行前提</div>
  <div style="color: #BF360C; line-height: 1.7;">
    ・Serverless SQL Warehouse M（<code>642919d1a269f9d0</code>）にアタッチ<br>
    ・所要時間: <b>15〜25 分</b>（Fact 5億行 + Dim 1億行×2 の生成 + OPTIMIZE + ANALYZE）<br>
    ・このノートブックは <b>一度だけ実行</b>すればOK
  </div>
</div>

## Step 1 / 7 · スキーマ作成

In [0]:
CREATE SCHEMA IF NOT EXISTS konomi_demo_catalog.star_schema_pruning_demo
COMMENT 'DCM Star Schema / File Pruning validation';

USE CATALOG konomi_demo_catalog;
USE SCHEMA star_schema_pruning_demo;
SELECT current_catalog() AS catalog, current_schema() AS schema;

## Step 2 / 7 · 補助 Dimension テーブル生成（Product / Time / Promotion）

In [0]:
-- dim_product: 10k 行
CREATE OR REPLACE TABLE dim_product (
  product_id BIGINT,
  product_name STRING,
  unit_price DECIMAL(10, 2),
  product_line STRING
)
COMMENT 'プロダクトディメンション';

INSERT INTO dim_product
SELECT
  id + 1 AS product_id,
  CONCAT('Product_', LPAD(CAST(id + 1 AS STRING), 6, '0')) AS product_name,
  CAST(100 + RAND() * 9900 AS DECIMAL(10, 2)) AS unit_price,
  CASE MOD(id, 5)
    WHEN 0 THEN 'Electronics' WHEN 1 THEN 'Grocery' WHEN 2 THEN 'Apparel'
    WHEN 3 THEN 'Home'        ELSE 'Toys'
  END AS product_line
FROM range(10000);

In [0]:
-- dim_time: 5 年分の日付
CREATE OR REPLACE TABLE dim_time (
  time_id BIGINT,
  order_date DATE,
  month INT,
  quarter INT,
  year INT
)
COMMENT '時間ディメンション（2021-01-01 〜 2025-12-31）';

INSERT INTO dim_time
SELECT
  id + 1 AS time_id,
  DATE_ADD(DATE '2021-01-01', CAST(id AS INT)) AS order_date,
  MONTH(DATE_ADD(DATE '2021-01-01', CAST(id AS INT))) AS month,
  QUARTER(DATE_ADD(DATE '2021-01-01', CAST(id AS INT))) AS quarter,
  YEAR(DATE_ADD(DATE '2021-01-01', CAST(id AS INT))) AS year
FROM range(1826);

In [0]:
-- dim_promotion: 100 行
CREATE OR REPLACE TABLE dim_promotion (
  promotion_id BIGINT,
  promo_name STRING,
  ad_type STRING,
  coupon_type STRING,
  price_reduction_type STRING
)
COMMENT 'プロモーションディメンション';

INSERT INTO dim_promotion
SELECT
  id + 1 AS promotion_id,
  CONCAT('Promo_', LPAD(CAST(id + 1 AS STRING), 3, '0')) AS promo_name,
  CASE MOD(id, 4) WHEN 0 THEN 'TV' WHEN 1 THEN 'Web' WHEN 2 THEN 'Flyer' ELSE 'SNS' END AS ad_type,
  CASE MOD(id, 3) WHEN 0 THEN 'Percent' WHEN 1 THEN 'Fixed' ELSE 'BOGO' END AS coupon_type,
  CASE MOD(id, 2) WHEN 0 THEN 'Discount' ELSE 'Bundle' END AS price_reduction_type
FROM range(100);

## Step 3 / 7 · Customer Dimension を 2 種類生成（1億行）

<div style="border-left: 4px solid #388E3C; background: #E8F5E9; padding: 14px 18px; border-radius: 4px;">
  <div style="font-weight: 700; color: #1B5E20; margin-bottom: 6px;">✅ ポイント: 大きな Dim で File Pruning を検証</div>
  <div style="color: #1B5E20; line-height: 1.7;">
    お客様の質問の背景は「Dim テーブルが大きいので、WHERE で絞ったときのスキャンコストが心配」。<br>
    そこで <b>Dim を 1億行</b> にして、以下 2 種類を用意し File Pruning の効果を対比します:<br>
    ・<b>dim_customer_clustered</b>: <code>CLUSTER BY (prefecture)</code> あり → pruning が効くべきパターン<br>
    ・<b>dim_customer_unclustered</b>: クラスタリングなし → pruning が効かないパターン
  </div>
</div>

In [0]:
-- dim_customer_clustered: 1億行、CLUSTER BY (prefecture)
CREATE OR REPLACE TABLE dim_customer_clustered (
  customer_id BIGINT,
  name STRING,
  city STRING,
  prefecture STRING,
  zip STRING
)
CLUSTER BY (prefecture)
COMMENT 'Customer dimension 100M rows, CLUSTER BY prefecture';

INSERT INTO dim_customer_clustered
SELECT
  id + 1 AS customer_id,
  CONCAT('Customer_', LPAD(CAST(id + 1 AS STRING), 9, '0')) AS name,
  CASE MOD(id, 10)
    WHEN 0 THEN '新宿区' WHEN 1 THEN '大阪市' WHEN 2 THEN '名古屋市' WHEN 3 THEN '札幌市'
    WHEN 4 THEN '福岡市' WHEN 5 THEN '仙台市' WHEN 6 THEN '横浜市'   WHEN 7 THEN '広島市'
    WHEN 8 THEN '京都市' ELSE '神戸市'
  END AS city,
  CASE MOD(id, 10)
    WHEN 0 THEN '東京都' WHEN 1 THEN '大阪府' WHEN 2 THEN '愛知県' WHEN 3 THEN '北海道'
    WHEN 4 THEN '福岡県' WHEN 5 THEN '宮城県' WHEN 6 THEN '神奈川県' WHEN 7 THEN '広島県'
    WHEN 8 THEN '京都府' ELSE '兵庫県'
  END AS prefecture,
  LPAD(CAST(MOD(id, 999) + 1 AS STRING), 3, '0') || '-' || LPAD(CAST(MOD(id, 9999) + 1 AS STRING), 4, '0') AS zip
FROM range(100000000);

In [0]:
-- dim_customer_unclustered: 1億行、クラスタリングなし
CREATE OR REPLACE TABLE dim_customer_unclustered (
  customer_id BIGINT,
  name STRING,
  city STRING,
  prefecture STRING,
  zip STRING
)
COMMENT 'Customer dimension 100M rows, NO clustering (comparison)';

INSERT INTO dim_customer_unclustered
SELECT
  id + 1 AS customer_id,
  CONCAT('Customer_', LPAD(CAST(id + 1 AS STRING), 9, '0')) AS name,
  CASE MOD(id, 10)
    WHEN 0 THEN '新宿区' WHEN 1 THEN '大阪市' WHEN 2 THEN '名古屋市' WHEN 3 THEN '札幌市'
    WHEN 4 THEN '福岡市' WHEN 5 THEN '仙台市' WHEN 6 THEN '横浜市'   WHEN 7 THEN '広島市'
    WHEN 8 THEN '京都市' ELSE '神戸市'
  END AS city,
  CASE MOD(id, 10)
    WHEN 0 THEN '東京都' WHEN 1 THEN '大阪府' WHEN 2 THEN '愛知県' WHEN 3 THEN '北海道'
    WHEN 4 THEN '福岡県' WHEN 5 THEN '宮城県' WHEN 6 THEN '神奈川県' WHEN 7 THEN '広島県'
    WHEN 8 THEN '京都府' ELSE '兵庫県'
  END AS prefecture,
  LPAD(CAST(MOD(id, 999) + 1 AS STRING), 3, '0') || '-' || LPAD(CAST(MOD(id, 9999) + 1 AS STRING), 4, '0') AS zip
FROM range(100000000);

## Step 4 / 7 · Fact テーブル生成（5億行、CLUSTER BY customer_id）

In [0]:
CREATE OR REPLACE TABLE fact_sales (
  transaction_id BIGINT,
  customer_id BIGINT,
  product_id BIGINT,
  time_id BIGINT,
  promotion_id BIGINT,
  revenue DECIMAL(12, 2),
  units_sold INT
)
CLUSTER BY (customer_id)
COMMENT '売上ファクトテーブル（5億行、Liquid Clustering by customer_id）';

INSERT INTO fact_sales
SELECT
  id + 1 AS transaction_id,
  CAST(MOD(ABS(HASH(id, 31)), 100000000) + 1 AS BIGINT) AS customer_id,
  CAST(MOD(ABS(HASH(id, 37)), 10000) + 1 AS BIGINT) AS product_id,
  CAST(MOD(ABS(HASH(id, 41)), 1826) + 1 AS BIGINT) AS time_id,
  CAST(MOD(ABS(HASH(id, 43)), 100) + 1 AS BIGINT) AS promotion_id,
  CAST(MOD(ABS(HASH(id, 47)), 1000000) / 100.0 AS DECIMAL(12, 2)) AS revenue,
  CAST(MOD(ABS(HASH(id, 53)), 10) + 1 AS INT) AS units_sold
FROM range(500000000);

## Step 5 / 7 · OPTIMIZE（Liquid Clustering を適用）

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 14px 18px; border-radius: 4px;">
  <div style="color: #0d47a1; line-height: 1.7;">
    <code>CLUSTER BY</code> を定義しただけでは物理配置は行われません。<br>
    <code>OPTIMIZE</code> を実行することで、指定キー近傍のレコードが同一ファイルに集約されます。
  </div>
</div>

In [0]:
OPTIMIZE dim_customer_clustered FULL;

In [0]:
OPTIMIZE fact_sales;

## Step 6 / 7 · 統計情報の収集

In [0]:
ANALYZE TABLE dim_customer_clustered   COMPUTE STATISTICS FOR ALL COLUMNS;

In [0]:
ANALYZE TABLE dim_customer_unclustered COMPUTE STATISTICS FOR ALL COLUMNS;

In [0]:
ANALYZE TABLE dim_product   COMPUTE STATISTICS FOR ALL COLUMNS;

In [0]:
ANALYZE TABLE dim_time      COMPUTE STATISTICS FOR ALL COLUMNS;

In [0]:
ANALYZE TABLE dim_promotion COMPUTE STATISTICS FOR ALL COLUMNS;

In [0]:
ANALYZE TABLE fact_sales    COMPUTE STATISTICS FOR ALL COLUMNS;

## Step 7 / 7 · 生成結果の確認

In [0]:
-- テーブル一覧と行数
SELECT 'dim_customer_clustered'   AS table_name, COUNT(*) AS row_count FROM dim_customer_clustered
UNION ALL SELECT 'dim_customer_unclustered', COUNT(*) FROM dim_customer_unclustered
UNION ALL SELECT 'dim_product',   COUNT(*) FROM dim_product
UNION ALL SELECT 'dim_time',      COUNT(*) FROM dim_time
UNION ALL SELECT 'dim_promotion', COUNT(*) FROM dim_promotion
UNION ALL SELECT 'fact_sales',    COUNT(*) FROM fact_sales
ORDER BY table_name;

In [0]:
-- Dim clustered のファイル数（少なく集約されているはず）
DESCRIBE DETAIL dim_customer_clustered;

In [0]:
-- Dim unclustered のファイル数（比較用、クラスタリングなし）
DESCRIBE DETAIL dim_customer_unclustered;

In [0]:
-- Fact のファイル数（customer_id で Liquid Clustering）
DESCRIBE DETAIL fact_sales;

In [0]:
-- 都道府県の分布（各県 10%）
SELECT prefecture, COUNT(*) AS customers, ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM dim_customer_clustered
GROUP BY prefecture
ORDER BY customers DESC;

<div style="background: linear-gradient(135deg, #388E3C 0%, #1B5E20 100%); color: white; padding: 20px 28px; border-radius: 8px; margin-top: 16px;">
  <div style="font-size: 20px; font-weight: 700; margin-bottom: 8px;">✅ Setup 完了</div>
  <div style="line-height: 1.7;">
    次は <b>02_pruning_verification</b> ノートブックを開いて File Pruning を検証しましょう。
  </div>
</div>